<a href="https://colab.research.google.com/github/saiful-cse/isp-user-behavior-anomaly-dataset/blob/main/User_Level_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import hashlib

# ==============================================================================
# OPTION A: DATA ANONYMIZATION PIPELINE (For Reference Only)
# This block demonstrates how the raw private ISP log was processed for privacy.
# ==============================================================================
"""
# 1. Load private raw ISP log
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/raw_isp_logs")

# 2. Hash username (irreversible)
def hash_username(u):
    return hashlib.sha256(u.encode()).hexdigest()[:12]

df["user_hash"] = df["username"].astype(str).apply(hash_username)

# 3. Create numeric user_id (for ML use)
df["user_id"] = df["user_hash"].astype("category").cat.codes

# 4. Drop sensitive identifiers (PII)
df_anonymized = df.drop(columns=["username", "src_ip", "dst_ip"])

# 5. Round timestamp to preserve temporal privacy
#df_anonymized["timestamp"] = pd.to_datetime(df_anonymized["timestamp"]).dt.floor("min")

# 6. Reorder columns
df_anonymized = df_anonymized[["timestamp", "user_id", "protocol", "src_port", "dst_port", "length"]]
# df_anonymized.to_csv("user_logs_anonymized_546278.csv", index=False)

output_path = "/content/drive/MyDrive/Colab Notebooks/user_logs_anonymized_546278_row.csv"
df_anonymized.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
"""

# ==============================================================================
# OPTION B: REPRODUCIBLE MODELING WITH PUBLIC ANONYMIZED DATASET
# Direct internet-based data loading via Google Drive direct download URL.
# ==============================================================================

# old anonnimize https://drive.google.com/file/d/1cOvjevbt3h2MKl4fKTNVAYrHtQCFtdA0/view?usp=drive_link
#file_id = "1cOvjevbt3h2MKl4fKTNVAYrHtQCFtdA0"


# Step 1: Set the Google Drive File ID for the publicly shared anonymized dataset
# (Replace 'YOUR_DRIVE_FILE_ID_HERE' with your actual shared file ID)
file_id = "1kiHEJ6hzMuoa48y3BS45Y1-N1-T_SkX9"
# https://drive.google.com/file/d/1kiHEJ6hzMuoa48y3BS45Y1-N1-T_SkX9/view?usp=drive_link
direct_download_url = f"https://docs.google.com/uc?export=download&id={file_id}"

# Step 2: Load the dataset directly into the workflow without mounting local drives
df = pd.read_csv(direct_download_url)

# Step 3: Ensure the timestamp column is correctly parsed as a datetime object
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Step 4: Verify and display the structural integrity of the loaded public dataset
print("Public dataset loaded successfully from Google Drive. Shape:")
print(df.head())



In [ ]:
#Cell:2,  Required Library Import
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Anomaly Detection Models
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Baseline Model
from sklearn.ensemble import RandomForestClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score
)

import warnings
warnings.filterwarnings("ignore")


In [ ]:
#cell:4, Basic Snatity Check
# Total event
print("Total events:", len(df))

# Total unique user
print("Unique users:", df['user_id'].nunique())


In [ ]:
#Cell:6, User-level aggregation  Event → User aggregation
user_df = df.groupby('user_id').agg(
    mean_packet_length=('length', 'mean'),
    max_packet_length=('length', 'max'),
    unique_src_ports=('src_port', 'nunique'),
    unique_dst_ports=('dst_port', 'nunique'),
    total_events=('user_id', 'count')
).reset_index()

user_df.head()


In [ ]:
#Cell:7, Feature Scalling Feature matrix তৈরি
X = user_df.drop(columns=['user_id'])

# Scaling করা (সব মডেলের জন্য একই scaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
#Cell:8, Autoencoder architecture
input_dim = X_scaled.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(8, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# Train
autoencoder.fit(
    X_scaled, X_scaled,
    epochs=30,
    batch_size=16,
    verbose=0
)

# Reconstruction error
reconstructions = autoencoder.predict(X_scaled)
mse = np.mean(np.square(X_scaled - reconstructions), axis=1)

# Threshold (example: 99th percentile)
ae_threshold = np.percentile(mse, 99)
ae_labels = (mse > ae_threshold).astype(int)


In [ ]:
#Cell:9, Isolation Forest
iso = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=40
)

iso_labels = iso.fit_predict(X_scaled)
iso_labels = np.where(iso_labels == -1, 1, 0)


In [ ]:
#Cell:10, One-Class SVM
svm = OneClassSVM(
    kernel='rbf',
    nu=0.01,
    gamma='scale'
)

svm_labels = svm.fit_predict(X_scaled)
svm_labels = np.where(svm_labels == -1, 1, 0)


In [ ]:
#Cell:11. Prepare Label Table
labels_df = user_df[['user_id']].copy()
labels_df['ae_anomaly'] = ae_labels
labels_df['if_anomaly'] = iso_labels
labels_df['svm_anomaly'] = svm_labels

labels_df.head()


In [ ]:
# ==========================================
# Detection Count Table for Individual and Consensus Models
# ==========================================

import pandas as pd

# ---- Step 1: Create Consensus Label (Majority Voting) ----
labels_df['consensus_label'] = (
    labels_df[['ae_anomaly', 'if_anomaly', 'svm_anomaly']]
    .sum(axis=1) >= 2
).astype(int)

# ---- Step 2: Total Users ----
total_users = len(labels_df)

# ---- Step 3: Detection Summary Function ----
def detection_summary(model_name, predictions):
    detected = predictions.sum()
    detection_rate = (detected / total_users) * 100
    return [model_name, total_users, detected, round(detection_rate, 2)]

# ---- Step 4: Create Table ----
detection_table = pd.DataFrame([
    detection_summary("Autoencoder", labels_df['ae_anomaly']),
    detection_summary("IsolationForest", labels_df['if_anomaly']),
    detection_summary("OneClassSVM", labels_df['svm_anomaly']),
    detection_summary("ProposedConsensus", labels_df['consensus_label'])
], columns=["Model", "Total Users", "Detected Anomalies", "Detection Rate (%)"])

detection_table


In [ ]:
# ==========================================
# Comprehensive Performance Comparison Table
# ==========================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score
)

# ---- Step 1: Consensus label তৈরি (majority voting) ----
# কমপক্ষে 2টা model যদি anomaly বলে → anomaly

labels_df['consensus_label'] = (
    labels_df[['ae_anomaly', 'if_anomaly', 'svm_anomaly']]
    .sum(axis=1) >= 2
).astype(int)

# ---- Step 2: Metric function ----
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "CohenKappa": cohen_kappa_score(y_true, y_pred)
    }

# ---- Step 3: Use consensus as reference ----
y_true = labels_df['consensus_label']

results = {
    "Autoencoder": compute_metrics(y_true, labels_df['ae_anomaly']),
    "IsolationForest": compute_metrics(y_true, labels_df['if_anomaly']),
    "OneClassSVM": compute_metrics(y_true, labels_df['svm_anomaly']),
    "ProposedConsensus": compute_metrics(y_true, labels_df['consensus_label'])
}

results_df = pd.DataFrame(results).T.round(3)

results_df


In [ ]:
# ==============================================
# Import required libraries
# ==============================================
from sklearn.metrics import confusion_matrix
import pandas as pd


# ==============================================
# Step 1: Ensure consensus label exists
# ==============================================
labels_df['consensus_label'] = (
    labels_df[['ae_anomaly', 'if_anomaly', 'svm_anomaly']]
    .sum(axis=1) >= 2
).astype(int)


# ==============================================
# Step 2: Define individual models
# ==============================================
models = {
    "Autoencoder": labels_df['ae_anomaly'],
    "IsolationForest": labels_df['if_anomaly'],
    "OneClassSVM": labels_df['svm_anomaly']
}

y_true = labels_df['consensus_label']


# ==============================================
# Step 3: Generate confusion matrix
# ==============================================
for name, y_pred in models.items():

    cm = confusion_matrix(y_true, y_pred)

    print(f"\nConfusion Matrix: {name} vs Consensus\n")

    cm_df = pd.DataFrame(
        cm,
        index=["Actual Normal (Consensus=0)", "Actual Anomaly (Consensus=1)"],
        columns=["Predicted Normal", "Predicted Anomaly"]
    )

    print(cm_df)


In [ ]:
#vertical ফিগার

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# ================================
# Step 1: Consensus label তৈরি
# ================================
labels_df['consensus_label'] = (
    labels_df[['ae_anomaly', 'if_anomaly', 'svm_anomaly']]
    .sum(axis=1) >= 2
).astype(int)

models = {
    "Autoencoder": labels_df['ae_anomaly'],
    "IsolationForest": labels_df['if_anomaly'],
    "OneClassSVM": labels_df['svm_anomaly']
}

y_true = labels_df['consensus_label']

# ================================
# Step 2: Confusion matrices calculate
# ================================
cms = [confusion_matrix(y_true, y_pred) for y_pred in models.values()]
vmin = 0
vmax = max(cm.max() for cm in cms)

# ================================
# Step 3: Vertical composite figure
# ================================
fig, axes = plt.subplots(3, 1, figsize=(6, 12))  # Narrow width for double column

for ax, cm, name in zip(axes, cms, models.keys()):

    # 🔴 এখানে colormap change করা যায়
    im = ax.imshow(cm, cmap="viridis", vmin=vmin, vmax=vmax)

    ax.set_title(name, fontsize=11)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Predicted Normal", "Predicted Anomaly"], fontsize=9)
    ax.set_yticklabels(["Actual Normal", "Actual Anomaly"], fontsize=9)

    threshold = vmax / 2
    for i in range(2):
        for j in range(2):
            color = "white" if cm[i, j] < threshold else "black"
            ax.text(j, i, cm[i, j],
                    ha="center",
                    va="center",
                    color=color,
                    fontsize=10)

# Layout adjustment
plt.subplots_adjust(hspace=0.5, right=0.85)

# ================================
# Step 4: Shared vertical colorbar
# ================================
cbar_ax = fig.add_axes([0.88, 0.15, 0.03, 0.7])
fig.colorbar(im, cax=cbar_ax)
cbar_ax.set_ylabel("Number of Samples")

# ================================
# Step 5: 300 DPI PNG Save
# ================================
plt.savefig("confusion_matrix_vertical_double_column.png",
            dpi=300,
            bbox_inches='tight')

plt.show()

# Download from Colab
from google.colab import files
files.download("confusion_matrix_vertical_double_column.png")

In [ ]:
#Horizontal ফিগার
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# ================================
# Step 1: Consensus label তৈরি
# ================================
labels_df['consensus_label'] = (
    labels_df[['ae_anomaly', 'if_anomaly', 'svm_anomaly']]
    .sum(axis=1) >= 2
).astype(int)

models = {
    "Autoencoder": labels_df['ae_anomaly'],
    "IsolationForest": labels_df['if_anomaly'],
    "OneClassSVM": labels_df['svm_anomaly']
}

y_true = labels_df['consensus_label']

# ================================
# Step 2: Confusion matrices calculate
# ================================
cms = [confusion_matrix(y_true, y_pred) for y_pred in models.values()]
vmin = 0
vmax = max(cm.max() for cm in cms)

# ================================
# Step 3: Composite figure তৈরি
# ================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, cm, name in zip(axes, cms, models.keys()):

    # 🔴 এখানে colormap change করা যায়।
    # Example: "viridis", "plasma", "inferno", "cividis", "magma"
    im = ax.imshow(cm, cmap="viridis", vmin=vmin, vmax=vmax)

    ax.set_title(name)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Predicted Normal", "Predicted Anomaly"])
    ax.set_yticklabels(["Actual Normal", "Actual Anomaly"])

    threshold = vmax / 2
    for i in range(2):
        for j in range(2):
            color = "white" if cm[i, j] < threshold else "black"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

plt.subplots_adjust(wspace=0.35, right=0.88)

# ================================
# Step 4: Shared colorbar
# ================================
cbar_ax = fig.add_axes([0.90, 0.18, 0.02, 0.65])
fig.colorbar(im, cax=cbar_ax)
cbar_ax.set_ylabel("Number of Samples")

# ================================
# Step 5: 300 DPI PNG Save
# ================================
plt.savefig("confusion_matrix_composite.png", dpi=300, bbox_inches='tight')

plt.show()

# Download from Colab
#from google.colab import files
#files.download("confusion_matrix_composite.png")


In [ ]:
#Result download as file
import os
os.listdir('/content/drive/MyDrive')


save_dir = "/content/drive/MyDrive/ISP_Anomaly_Results"
os.makedirs(save_dir, exist_ok=True)


#results_df.to_csv(f"{save_dir}/overall_results.csv", index=True)
#labels_df.to_csv(f"{save_dir}/user_level_labels.csv", index=False)

#print("Files saved at:", save_dir)


In [ ]:
# ==========================================
# Cell-1: Figure 6.2 – Comparative Performance (FINAL FIXED)
# ==========================================

import matplotlib.pyplot as plt
import numpy as np

plt.close('all')

# Models & metrics from results_df
models = results_df.index.tolist()

accuracy = results_df["Accuracy"].values
precision = results_df["Precision"].values
recall = results_df["Recall"].values
f1 = results_df["F1-score"].values
kappa = results_df["CohenKappa"].values

x = np.arange(len(models))
width = 0.15

plt.figure(figsize=(10, 6))

# Bars (same style as your sample)
plt.bar(x - 2*width, accuracy, width, label='Accuracy')
plt.bar(x - width,  precision, width, label='Precision')
plt.bar(x,          recall,    width, label='Recall')
plt.bar(x + width,  f1,        width, label='F1-score')
plt.bar(x + 2*width,kappa,     width, label="Cohen's Kappa")

# Axis formatting
plt.xlabel('Anomaly Detection Models', fontsize=12)
plt.ylabel('Performance Score', fontsize=12)
plt.xticks(x, models, rotation=0)

# 🔑 CRITICAL FIX: full visible range
plt.ylim(0.0, 1.08)

# Legend (unchanged)
plt.legend(
    loc='upper right',
    bbox_to_anchor=(0.98, 0.98),
    frameon=True,
    fontsize=10,
    facecolor='white',
    edgecolor='gray'
)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

#Step 1: Save BEFORE plt.show()
plt.savefig("Figure_6_2_Comparative_Performance.png", dpi=300, bbox_inches='tight')

plt.show()

# Step 3: Download from Colab
#from google.colab import files
#files.download("Figure_6_2_Comparative_Performance.png")


In [ ]:
# ================================
# Install SHAP (if not installed)
# ================================
!pip install shap

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ================================
# Prepare feature matrix
# ================================

# তোমার user-level aggregated features
feature_cols = [
    "total_events",
    "unique_src_ports",
    "unique_dst_ports",
    "mean_packet_length",
    "max_packet_length"
]

X = user_df[feature_cols]

# ================================
# Use trained Isolation Forest model
# ================================
# ধরে নিচ্ছি iso_model already trained আছে

explainer = shap.TreeExplainer(iso)

shap_values = explainer.shap_values(X)

# ================================
# SHAP Summary Plot
# ================================

plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("shap_summary_isolation_forest.png", dpi=300)
plt.show()

# Step 3: Download from Colab
#from google.colab import files
#files.download("shap_summary_isolation_forest.png")


In [ ]:
%who


In [ ]:
# =========================================
# Install LIME (if not installed)
# =========================================
!pip install lime

import numpy as np
import matplotlib.pyplot as plt
from lime.lime_tabular import LimeTabularExplainer

# =========================================
# Step 1: Feature matrix প্রস্তুত
# =========================================
feature_cols = [
    "total_events",
    "unique_src_ports",
    "unique_dst_ports",
    "mean_packet_length",
    "max_packet_length"
]

X = user_df[feature_cols].values

# =========================================
# Step 2: Consensus anomaly থেকে একটি user নির্বাচন
# =========================================
anomaly_index = labels_df[labels_df['consensus_label'] == 1].index[0]

instance = user_df.loc[anomaly_index, feature_cols].values

# =========================================
# Step 3: Isolation Forest decision wrapper
# =========================================
# LIME probability output চায়, তাই decision_function normalize করা হবে

def predict_proba(data):
    scores = iso.decision_function(data)

    # normalize to 0-1 range
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min())

    # anomaly probability inverse (lower score = more anomalous)
    anomaly_prob = 1 - scores_norm

    # return দুই কলাম: [normal_prob, anomaly_prob]
    return np.vstack((scores_norm, anomaly_prob)).T

# =========================================
# Step 4: LIME Explainer তৈরি
# =========================================
explainer = LimeTabularExplainer(
    training_data=X,
    feature_names=feature_cols,
    class_names=["Normal", "Anomaly"],
    mode="classification"
)

# =========================================
# Step 5: Explanation generate
# =========================================
exp = explainer.explain_instance(
    instance,
    predict_proba,
    num_features=5
)

# =========================================
# Step 6: Figure save (300 DPI)
# =========================================
fig = exp.as_pyplot_figure()
plt.tight_layout()
plt.title("")
plt.savefig("lime_local_explanation.png", dpi=300, bbox_inches='tight')
plt.show()

# Step 3: Download from Colab
from google.colab import files
files.download("lime_local_explanation.png")
